# Downstream Task Examples

This notebook demonstrates two downstream tasks using pre-trained BulkRNABERT representations:

1. **Cancer type classification** — predict cancer cohort (5 TCGA cohorts) from RNA-seq
2. **Pan-cancer survival** — predict relative survival risk from RNA-seq

Both models share the same BulkRNABERT encoder backbone.

In [ ]:
import pickle

import haiku as hk
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from multiomics_open_research.bulk_rna_bert.downstream.pretrained import get_pretrained_downstream_model
from multiomics_open_research.common.preprocess import preprocess_omic

In [ ]:
rna_seq_df = pd.read_csv("../data/bulkrnabert/tcga_sample.csv")
print(f"{len(rna_seq_df)} samples")
rna_seq_df[["identifier", "survival_time", "event"]]

## 1. Cancer Type Classification

Predicts one of 5 TCGA cancer cohorts from RNA-seq expression.

In [ ]:
parameters, forward_fn, tokenizer, config, mlm_config = get_pretrained_downstream_model(
    model_name="tcga_5_cohorts",
    checkpoint_directory="../checkpoints/",
)
forward_fn = hk.transform(forward_fn)

In [ ]:
rna_seq_array = preprocess_omic(rna_seq_df, mlm_config)
tokens_ids = tokenizer.batch_tokenize(rna_seq_array)
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

random_key = jax.random.PRNGKey(0)
outs = forward_fn.apply(parameters, random_key, tokens[:1])

In [ ]:
with open("../data/bulkrnabert/5_cohorts_labels_mapping.pkl", "rb") as f:
    label_mapping = pickle.load(f)

predicted_cancer_type = label_mapping[int(outs["logits"].argmax())]
print(f"Cancer type prediction for {rna_seq_df['identifier'].iloc[0]}: {predicted_cancer_type}")

## 2. Pan-Cancer Survival

Predicts a **log partial hazard** (Cox linear predictor) for each sample. 
A higher log partial hazard score indicates a higher predicted risk of death, and therefore a shorter expected survival time.

In [ ]:
surv_parameters, surv_forward_fn, surv_tokenizer, surv_config, surv_mlm_config = get_pretrained_downstream_model(
    model_name="tcga_pancancer_survival",
    checkpoint_directory="../checkpoints/",
)
surv_forward_fn = hk.transform(surv_forward_fn)

In [ ]:
surv_df = rna_seq_df.loc[
    [rna_seq_df["survival_time"].idxmin(),
     rna_seq_df["survival_time"].idxmax()]
].reset_index(drop=True)

surv_array = preprocess_omic(surv_df, surv_mlm_config)
surv_tokens_ids = surv_tokenizer.batch_tokenize(surv_array)
surv_tokens = jnp.asarray(surv_tokens_ids, dtype=jnp.int32)

surv_outs = surv_forward_fn.apply(surv_parameters, random_key, surv_tokens)
log_hazard_scores = surv_outs["logits"].squeeze(-1)

In [ ]:
surv_results = pd.DataFrame({
    "identifier": surv_df["identifier"],
    "observed_survival_days": surv_df["survival_time"],
    "event": surv_df["event"],
    "log_hazard_score": np.array(log_hazard_scores),
}).sort_values("log_hazard_score", ascending=False).reset_index(drop=True)
surv_results

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 5))
ax2 = ax1.twinx()

x = np.arange(len(surv_results))
width = 0.35

bars_risk = ax1.bar(x - width / 2, surv_results["log_hazard_score"], width,
                    label="Predicted risk (log partial hazard)", color="#C44E52", alpha=0.85)
bars_surv = ax2.bar(x + width / 2, surv_results["observed_survival_days"], width,
                    label="Observed survival (days)", color="#4C72B0", alpha=0.85)

for bar, score in zip(bars_risk, surv_results["log_hazard_score"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{score:.2f}",
             ha="center", fontsize=9, color="#C44E52")
for bar, days in zip(bars_surv, surv_results["observed_survival_days"]):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, f"{int(days)}d",
             ha="center", fontsize=9, color="#4C72B0")

ax1.set_xticks(x)
ax1.set_xticklabels(surv_results["identifier"], fontsize=9)
ax1.set_ylabel("Log partial hazard", color="#C44E52")
ax1.tick_params(axis="y", labelcolor="#C44E52")
ax2.set_ylabel("Observed survival (days)", color="#4C72B0")
ax2.tick_params(axis="y", labelcolor="#4C72B0")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper center")

ax1.set_title("Predicted risk vs observed survival — higher risk should correspond to shorter survival")
plt.tight_layout()
plt.show()